# Preprocesamiento de datos

## Objetivo del notebook

El objetivo de este notebook es preparar el conjunto de datos para el entrenamiento de modelos de Machine Learning. Para ello se realizará la separación entre variables predictoras y variable objetivo, la división del conjunto de datos en entrenamiento y prueba y la construcción del pipeline de preprocesamiento.

Al finalizar este notebook se obtendrá un conjunto de datos preparado para el entrenamiento y evaluación de modelos de clasificación.

# 1. Importación de librerías

In [16]:
# Manipulación de datos
import pandas as pd
import numpy as np

# Visualización (opcional)
import matplotlib.pyplot as plt
import seaborn as sns

# Preprocesamiento y partición de datos
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline

# 2. Carga de datos

## Objetivo

En este apartado se carga el conjunto de datos obtenido tras el proceso de limpieza realizado en el Notebook 1. Este será el conjunto de datos utilizado durante el preprocesamiento y la preparación de los datos para el entrenamiento de los modelos de Machine Learning. 

In [17]:
# Carga del conjunto de datos procesado
df = pd.read_csv("../data/processed/telco_limpio.csv")
df.head()

,ID_Cliente,Genero,Adulto_Mayor,Tiene_Pareja,Tiene_Dependientes,Antiguedad_Meses,Servicio_Telefono,Multiples_Lineas,Servicio_Internet,Seguridad_Online,...,Proteccion_Dispositivo,Soporte_Tecnico,Streaming_TV,Streaming_Peliculas,Tipo_Contrato,Factura_Electronica,Metodo_Pago,Cargo_Mensual,Cargos_Totales,Abandono
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.50,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


# 3. Separación de variables predictoras y variable objetivo

## Objetivo

Antes de entrenar cualquier modelo de Machine Learning es necesario separar las variables predictoras de la variable objetivo. Las variables predictoras (`X`) contienen la información que utilizará el modelo para realizar las predicciones, mientras que la variable objetivo (`y`) representa el resultado que se desea predecir.

En este proyecto, la variable objetivo es **Abandono**, ya que el objetivo consiste en predecir si un cliente abandonará o no la compañía.

In [18]:
# Variables predictoras  
X = df.drop(columns=["ID_Cliente", "Abandono"])

# Variable objetivo
y = df["Abandono"]  # Variable objetivo (columna "Abandono")

In [19]:
print(f"Variables predictoras: {X.shape}")
print(f"Variable objetivo: {y.shape}")

Variables predictoras: (7043, 19)
Variable objetivo: (7043,)


# 4. División del conjunto de entrenamiento y prueba

## Objetivo

Antes de aplicar cualquier transformación a los datos, es necesario dividir el conjunto de datos en un conjunto de entrenamiento y otro de prueba. Esta separación permite entrenar los modelos utilizando únicamente los datos de entrenamiento y evaluar posteriormente su rendimiento sobre datos que el modelo nunca ha visto.

Realizar esta división antes del preprocesamiento evita el **data leakage**, garantizando una evaluación más realista y fiable del modelo.

In [20]:
# División del conjunto de datos en entrenamiento y prueba
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2, 
    random_state=42,
    stratify=y # Mantener la proporción de clases en ambos conjuntos
)   

In [21]:
print(f"X_train: {X_train.shape}")
print(f"X_test: {X_test.shape}")
print(f"y_train: {y_train.shape}")
print(f"y_test: {y_test.shape}")

X_train: (5634, 19)
X_test: (1409, 19)
y_train: (5634,)
y_test: (1409,)


# 5. Identificación de variables numéricas y categóricas

## Objetivo

Antes de construir el pipeline de preprocesamiento es necesario identificar qué variables son numéricas y cuáles son categóricas. Esta clasificación permitirá aplicar las transformaciones adecuadas a cada tipo de dato, como el escalado de las variables numéricas o la codificación de las variables categóricas.

Realizar esta separación facilita la construcción de un proceso de preprocesamiento automatizado, reproducible y fácilmente reutilizable en diferentes modelos de Machine Learning.

In [22]:
# Variables numéricas
variables_numericas = X_train.select_dtypes(
    include=["int64", "float64"]
).columns.tolist()

# Variables categóricas
variables_categoricas = X_train.select_dtypes(
    include=["object", "string"]
).columns.tolist()

In [23]:
print("Variables numéricas:")
print(variables_numericas)

print("\nVariables categóricas:")
print(variables_categoricas)

Variables numéricas:
['Adulto_Mayor', 'Antiguedad_Meses', 'Cargo_Mensual', 'Cargos_Totales']

Variables categóricas:
['Genero', 'Tiene_Pareja', 'Tiene_Dependientes', 'Servicio_Telefono', 'Multiples_Lineas', 'Servicio_Internet', 'Seguridad_Online', 'Copia_Seguridad_Online', 'Proteccion_Dispositivo', 'Soporte_Tecnico', 'Streaming_TV', 'Streaming_Peliculas', 'Tipo_Contrato', 'Factura_Electronica', 'Metodo_Pago']


# 6. Construcción del preprocesador

## Objetivo

Las variables numéricas y categóricas requieren transformaciones diferentes antes de ser utilizadas por un modelo de Machine Learning. En este apartado se construirá un preprocesador utilizando `ColumnTransformer`, que permitirá aplicar automáticamente el escalado a las variables numéricas y la codificación a las variables categóricas.

Este enfoque facilita la creación de un flujo de trabajo reproducible, reduce la posibilidad de errores y evita el data leakage al integrarse posteriormente dentro de un pipeline.

In [24]:
# Construcción del preprocesador
preprocesador = ColumnTransformer(
    transformers=[
        (
            "numericas",
            StandardScaler(),
            variables_numericas
        ),
        (
            "categoricas",
            OneHotEncoder(handle_unknown="ignore"),
            variables_categoricas
        )
    ]
)

# 7. Aplicación del preprocesamiento

## Objetivo

Una vez construido el preprocesador, se aplicarán las transformaciones sobre los conjuntos de entrenamiento y prueba.

El preprocesador se ajustará (**fit**) únicamente con los datos de entrenamiento y posteriormente se utilizará para transformar (**transform**) tanto el conjunto de entrenamiento como el de prueba. Este procedimiento evita el *data leakage* y garantiza que el modelo no utilice información procedente del conjunto de prueba durante el entrenamiento.

In [25]:
# Ajustar el preprocesador con los datos de entrenamiento
X_train_preprocesado = preprocesador.fit_transform(X_train)

# Aplicar las mismas transformaciones al conjunto de prueba
X_test_preprocesado = preprocesador.transform(X_test)

In [26]:
print(f"X_train preprocesado: {X_train_preprocesado.shape}")
print(f"X_test preprocesado: {X_test_preprocesado.shape}")

X_train preprocesado: (5634, 45)
X_test preprocesado: (1409, 45)


## Comprobación del preprocesamiento

Una vez aplicadas las transformaciones, resulta útil comprobar el número de variables generadas y conocer sus nombres. Esta verificación permite asegurar que el preprocesamiento se ha realizado correctamente y facilita la interpretación posterior de los modelos de Machine Learning.

In [27]:
# Obtener los nombres de las variables tras el preprocesamiento
nombres_variables = preprocesador.get_feature_names_out()

In [28]:
# Convertir el conjunto de entrenamiento preprocesado en un DataFrame
X_train_preprocesado_df = pd.DataFrame(
    X_train_preprocesado,
    columns=nombres_variables,
    index=X_train.index
)

In [29]:
X_train_preprocesado_df.head()

,numericas__Adulto_Mayor,numericas__Antiguedad_Meses,numericas__Cargo_Mensual,numericas__Cargos_Totales,categoricas__Genero_Female,categoricas__Genero_Male,categoricas__Tiene_Pareja_No,categoricas__Tiene_Pareja_Yes,categoricas__Tiene_Dependientes_No,categoricas__Tiene_Dependientes_Yes,...,categoricas__Streaming_Peliculas_Yes,categoricas__Tipo_Contrato_Month-to-month,categoricas__Tipo_Contrato_One year,categoricas__Tipo_Contrato_Two year,categoricas__Factura_Electronica_No,categoricas__Factura_Electronica_Yes,categoricas__Metodo_Pago_Bank transfer (automatic),categoricas__Metodo_Pago_Credit card (automatic),categoricas__Metodo_Pago_Electronic check,categoricas__Metodo_Pago_Mailed check
3738,-0.441773,0.102371,-0.521976,-0.262257,0.0,1.0,1.0,0.0,1.0,0.0,...,1.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0
3151,-0.441773,-0.711743,0.337478,-0.503635,0.0,1.0,0.0,1.0,0.0,1.0,...,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0
4860,-0.441773,-0.793155,-0.809013,-0.749883,0.0,1.0,0.0,1.0,0.0,1.0,...,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,1.0
3867,-0.441773,-0.263980,0.284384,-0.172722,1.0,0.0,0.0,1.0,1.0,0.0,...,1.0,0.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,0.0
3810,-0.441773,-1.281624,-0.676279,-0.989374,0.0,1.0,0.0,1.0,0.0,1.0,...,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0


In [32]:
print(f"Número de variables originales: {X_train.shape[1]}")
print(f"Número de variables tras el preprocesamiento: {X_train_preprocesado_df.shape[1]}")

Número de variables originales: 19
Número de variables tras el preprocesamiento: 45


# 8. Conclusiones

En este notebook se ha preparado el conjunto de datos para el entrenamiento de modelos de Machine Learning. Para ello se ha separado la variable objetivo de las variables predictoras, se ha realizado la división entre los conjuntos de entrenamiento y prueba y se ha construido un pipeline de preprocesamiento utilizando `ColumnTransformer`.

Las variables numéricas han sido escaladas mediante `StandardScaler`, mientras que las variables categóricas se han transformado utilizando `OneHotEncoder`. Todas las transformaciones se han ajustado únicamente con el conjunto de entrenamiento, evitando así el data leakage y garantizando un flujo de trabajo reproducible.

Como resultado, se dispone de un conjunto de datos completamente preprocesado y preparado para el entrenamiento y evaluación de distintos modelos de clasificación.

# 9. Exportación de los datos preprocesados

## Objetivo

Con el fin de mantener una estructura modular del proyecto, se exportan los conjuntos de entrenamiento y prueba una vez finalizado el preprocesamiento. Estos archivos serán utilizados posteriormente en el notebook de modelado, evitando repetir el proceso de preparación de los datos y facilitando un flujo de trabajo más limpio y reproducible.

In [34]:
# Exportar los conjuntos de datos preprocesados

pd.DataFrame(
    X_train_preprocesado,
    columns=nombres_variables,
    index=X_train.index
).to_csv(
    "../data/processed/X_train_preprocesado.csv",
    index=False
)

pd.DataFrame(
    X_test_preprocesado,
    columns=nombres_variables,
    index=X_test.index
).to_csv(
    "../data/processed/X_test_preprocesado.csv",
    index=False
)

y_train.to_csv(
    "../data/processed/y_train.csv",
    index=False
)

y_test.to_csv(
    "../data/processed/y_test.csv",
    index=False
)

print("Conjuntos de datos exportados correctamente.")

Conjuntos de datos exportados correctamente.
